#Part 1 of the Zingg notebook
## It is responsible for initializing the Zingg environment, which includes the following steps:
- **Environment Setup:** Loads all necessary libraries and dependencies required for Zingg to run.
- **Path Setup:** Defines and sets up all relevant file paths, such as model directory, input data locations, output directories.
- **Performance Tuning:** Applies Spark and Zingg performance-related configurations, such as partitions, sample size, stopwords removal to optimize the execution of data processing tasks.

## Example Notebook For Training and Running Zingg Enterprise Entity Resolution Workflow on Databricks
This notebook runs the Zingg Febrl Example on Databricks. Please refer to the

- Enterprise Zingg Python API
- Zingg Official Documentation for details.

_This notebook has been tested on 16.4 LTS DBR version (Spark 3.5.2, scala 2.12)_

## Create a Spark Cluster and Install Zingg
# 
- Go to the Clusters tab, hit Create Cluster, and give it a name like “Zingg-Enterprise.”
- Set the runtime version to a current LTS (Long-Term Support) version for compatibility.
- Next, you’ll need to install Zingg. For this, we will be need the latest Zingg JAR file and the license.
- Create a Volume (managed volume) inside the schema and add the zingg-opensource-spark-0.8.0.jar and the zingg_license.jar to it.
- Upload the file: Open the cluster details, navigate to the Libraries section, and click Install New.
- Select the Volumes option and upload JAR from the specific path -> /Volumes/catalog_name/schema_name/volume_name/path_to_file (Zingg JAR)



### Please execute each cell one by one as per the instructions provided.


##Install Zingg

In [ ]:
%pip install zingg==0.6.0
dbutils.library.restartPython()


In [ ]:
%pip show zingg

In [0]:
## You can change these to the locations of your choice

## These are the only two settings that need to change

zinggDir = "/models"

modelId = "zinggTestApr28"

## Define locations for the model
The Zingg models and training data are persisted in dbfs.

Please edit the model id in the cell below to reflect your model.

In [ ]:
MARKED_DIR = zinggDir + "/" + modelId + "/trainingData/marked/"
UNMARKED_DIR = zinggDir + "/" + modelId + "/trainingData/unmarked/"

import pandas as pd
import numpy as np
 
import time
import uuid
 
from ipywidgets import widgets, interact, GridspecLayout
import base64
import pyspark.sql.functions as fn

##this code sets up the Zingg Python interface
from zingg.client import *
from zingg.pipes import *


def count_labeled_pairs(marked_pd):
  '''
  The purpose of this function is to count the labeled pairs in the marked folder.
  '''
  n_total = len(np.unique(marked_pd['z_cluster']))
  n_positive = len(np.unique(marked_pd[marked_pd['z_isMatch']==1]['z_cluster']))
  n_negative = len(np.unique(marked_pd[marked_pd['z_isMatch']==0]['z_cluster']))
  n_uncertain = len(np.unique(marked_pd[marked_pd['z_isMatch'] == 2]['z_cluster']))

  return n_positive, n_negative, n_uncertain, n_total

## Set the Directories path

## Start building the Zingg program
The following cell sets up the initial arguments for Zingg.

In [0]:

#build the arguments for zingg
args = Arguments()
# Set the modelid and the zingg dir. You can use this as is
args.setModelId(modelId)
args.setZinggDir(zinggDir)

## Performance settings
The numPartitions define how data is split across the cluster. Please change this as per your data and cluster size 

For details, refer to [Zingg performance tuning documentation](https://docs.zingg.ai/zingg/stepbystep/configuration/tuning-label-match-and-link-jobs).
In general:
- keep `numPartitions` to ~20-30x the worker vCPU count 
- Disable Spark's Adaptive Query Execution

__NOTE__: *Please modify this for your use case*

In [0]:
args.setNumPartitions(4)
args.setLabelDataSampleSize(0.5)


In [0]:
spark.conf.set("spark.sql.adaptive.enabled", False)

## Define the input
Please refer to [Pipes](https://docs.zingg.ai/latest/connectors/pipes) for details on different formats.

Please modify this for your data.

In [0]:
## Set table as the input delta source with format catalog_name.schema_name.table_name
## Please change this as required
table = "zingg_catalog.input.test"
inputPipe = UCPipe("testFebrl65", table)
args.setData(inputPipe)

In [ ]:
#show table
df = spark.table(table)
display(df)

## Configure the output
Here we configure the output format

Please modify this for your data.

In [0]:
## Set output table as the table for storing Zingg results with format catalog_name.schema_name.table_name 
## Please change this as required
outputTable = "zingg_catalog.output.febrlOutput28Apr"
outputPipe = UCPipe("resultFebrl", outputTable)
args.setOutput(outputPipe)

## Configure the statistics output path
Here we configure the stats path

Please make sure the path/name contains the placeholder "**_$ZINGG_DYNAMIC_STAT_NAME_**"

## Define the match fields, types, and strategies

The cell below is used to configure Zingg with the fields for use in matching and the match types.
Details on the field definitions can be found at [Zingg official docs](https://docs.zingg.ai/zingg/stepbystep/configuration/field-definitions)

__UPDATE__: The order in which you specify the columns in `field_defs` matters!  Put the most important fields first!

__NOTE__: *Please modify this for your use case*

In [0]:
recId = FieldDefinition("recId", "STRING", MatchType.DONT_USE)
fName = FieldDefinition("fName", "STRING", MatchType.FUZZY)
lName = FieldDefinition("lName", "STRING", MatchType.FUZZY)
streetId = FieldDefinition("streetId", "STRING", MatchType.DONT_USE)
street = FieldDefinition("street", "STRING", MatchType.FUZZY)
locality = FieldDefinition("locality", "STRING", MatchType.FUZZY)
area = FieldDefinition("area", "STRING", MatchType.FUZZY)
areaCode = FieldDefinition("areaCode", "STRING", MatchType.FUZZY)
state = FieldDefinition("state", "STRING", MatchType.FUZZY)
dob = FieldDefinition("dob", "STRING", MatchType.FUZZY)
ssn = FieldDefinition("ssn", "STRING", MatchType.EXACT)

args.setFieldDefinition([recId, fName, lName, streetId, street, locality, area, areaCode, state, dob, ssn])   